# 실기 대비
# 실전 문제풀이
# set 1

## 1) 데이터 및 시나리오

### 시중 TV 제품별 분석

> 시중 판매 중인 TV정보를 온라인 쇼핑몰 웹페이지에서 크롤링 하여 분석을 하고자 한다. 해당 데이터를 기반으로 각 TV의 제원은 어떤 한지 제원에 따른 평가는 어떤 한지 알아보고 향후 신제품 출시에 참고하고자 한다.

### 데이터 개요

| 파일명 | 행 | 열 | 인코딩 |
|---|---:|---:|---|
| `TV.csv` | 666 | 11 | UTF-8 |

### 변수 상세

| 변수명 | 유형 | 설명 |
|---|---|---|
| `Product_Name` | string | 상품명 |
| `Stars` | float | 평가 평균 점수 |
| `Ratings` | int | 평가 수 |
| `Reviews` | int | 후기 수 |
| `current_price` | int | 현재 가격 |
| `MRP` | int | 공장 출고가 |
| `channel` | string | 프리미엄 채널 서비스 제공 목록 |
| `Operating_system` | string | 운영체제 |
| `Picture_quality` | string | 해상도 |
| `Speaker` | string | 스피커 |
| `Frequency` | string | 주사율 |

## 2) 문제

### 필요 라이브러리 함수 및 클래스 목록

| 목록 |
|---|
| `from sklearn.ensemble import RandomForestRegressor` |

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

In [3]:
df = pd.read_csv('../../dataset/TV.csv')
#SCDI
display(df.shape)
display(df.columns)
display(df.dtypes)
display(df.isna().sum())

(666, 11)

Index(['Product_Name', 'Stars', 'Ratings', 'Reviews', 'current_price', 'MRP',
       'channel', 'Operating_system', 'Picture_quality', 'Speaker',
       'Frequency'],
      dtype='object')

Product_Name         object
Stars               float64
Ratings               int64
Reviews               int64
current_price         int64
MRP                   int64
channel              object
Operating_system     object
Picture_quality      object
Speaker              object
Frequency            object
dtype: object

Product_Name        0
Stars               0
Ratings             0
Reviews             0
current_price       0
MRP                 0
channel             0
Operating_system    0
Picture_quality     0
Speaker             0
Frequency           0
dtype: int64

### Q01.

주사율(`Frequency`)은 TV 제원 중 가격에 큰 영향을 미치는 제원이다.  
그런데 데이터 수집 중 실수가 발생하여 주사율(`Frequency`) 값 중 일부가 해상도(`Picture_quality`), 스피커(`Speaker`) 변수에 잘못 입력된 것이 발견되었다.  
세 변수에 들어간 주사율 데이터를 취합하여 전체 TV 중에서 주사율이 60Hz인 TV는 총 몇 대 인가?

※ 주사율은 2~3자리 숫자 뒤에 반드시 `Hz`라는 단위로 표기되어 있다.  
※ 세 변수 모두 주사율 정보가 없는 경우 결측치로 간주하고 이를 분석에서 제외하시오. `(정답 예시: 12)`

In [34]:
df_q1 = df.copy()
cols_q1 = ['Picture_quality', 'Speaker','Frequency']

display(df_q1[cols_q1].dtypes)
df_q1['f-p-s'] = df_q1['Frequency'] + ' ' + df_q1['Picture_quality'] + ' ' + df_q1['Speaker']
display(df_q1['f-p-s'])
df_cond_q1 = df_q1['f-p-s'].str.extract(r'(\d{2,3})\s*Hz')

#방법 1
display(df_cond_q1.value_counts())
display(df_cond_q1.isin(['60']).sum())

#방법 2
df_cond_q1 = df_cond_q1.rename(columns={0:'freq'})
display((df_cond_q1['freq'] == '60').sum())

Picture_quality    object
Speaker            object
Frequency          object
dtype: object

0      1 Year Warranty 60 Hz Refresh Rate 2 x HDMI | ...
1      60 Hz Refresh Rate HD Ready 1366 x 768 Pixels ...
2      50 Hz Refresh Rate HD Ready 1366 x 768 Pixels ...
3      60 Hz Refresh Rate HD Ready 1366 x 768 Pixels ...
4      60 Hz Refresh Rate HD Ready 1366 x 768 Pixels ...
                             ...                        
661         1 YEAR 60 Hz Refresh Rate 2 x HDMI | 1 x USB
662    50 Hz Refresh Rate Ultra HD (4K) 3840 x 2160 P...
663    3 Years Warranty 60 Hz Refresh Rate 2 x HDMI |...
664    2 x HDMI | 2 x USB 16 Speaker Output 50 Hz Ref...
665         1 YEAR 50 Hz Refresh Rate 2 x HDMI | 2 x USB
Name: f-p-s, Length: 666, dtype: object

60     510
50      69
100     30
120     30
200     19
800      2
300      1
58       1
dtype: int64

0    510
dtype: int64

510

### Q02.

TV의 해상도(`Picture_quality`)는 HD, 4K, 8K로 나뉜다. 최근 시장에 등장한 8K 제품군은 상대적으로 비싸 비교적 매출이 떨어지는 편이다. 하지만 8K 제품군 구매자가 4K 제품군 구매자 대비 얼마나 만족하고 있는지 확인해보고자 한다.  
8K 제품군의 평가 평균 점수(`Stars`)의 평균값과 4K 제품군의 평가 평균 점수와의 평균값 차이의 절대값을 산출하시오.

※ 화면 해상도는 관련 변수에서 `"HD"`, `"4K"`, `"8K"`로 찾아낼 수 있다.  
※ 화면 해상도 정보는 `"Operating_system"`, `"channel"`, `"Picture_quality"` 변수에 흩어져 있다.  
※ 결과는 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [47]:
df_q2 = df.copy()
cols_q2 = ['Operating_system', 'channel', 'Picture_quality']
df_q2['o-c-p'] = df_q2['Operating_system'] + ' ' + df_q2['channel'] + ' ' + df_q2['Picture_quality'] + ' '

cond_4k = df_q2['o-c-p'].str.contains('4K')
display(cond_4k.value_counts())
res_4k = df_q2.loc[cond_4k, :]['Stars'].mean()
display(res_4k)

cond_8k = df_q2['o-c-p'].str.contains('8K')
display(cond_8k.value_counts())
res_8k = df_q2.loc[cond_8k, :]['Stars'].mean()
display(res_8k)

display(round(abs(res_4k - res_8k),2))

True     365
False    301
Name: o-c-p, dtype: int64

3.2180821917808218

False    665
True       1
Name: o-c-p, dtype: int64

3.6

0.38

### Q03.

좋은 평가를 받는 제품의 조건을 알아보고자 한다. 이를 위해 기존 변수와 더불어 여러 파생변수를 생성 후 해당 변수들을 독립변수로 하고 평가 평균 점수(`Stars`)를 종속변수로 하여 Random Forest 분석을 통해 좋은 평가를 받는 제품의 조건을 알아보고자 한다.  
아래 절차를 수행하여 변수 중요도를 확인하고, 그 값 중 가장 큰 값을 가진 변수명을 기술하시오.

| 구분 | 변수 |
|---|---|
| 독립변수 | 후기 작성 비율: 후기 수(`Reviews`) / 평가 수(`Ratings`) |
| . | 공장 출고가(`MRP`) |
| . | 할인율: 현재 판매가(`current_price`) / 공장 출고가(`MRP`) |
| . | Netflix 제공 여부: 제공(1), 미제공(0) |
| . | Prime Video 제공 여부: 제공(1), 미제공(0) |
| . | 고해상도 여부: 4K 또는 8K(1), 나머지 0 |
| 종속변수 | 평가 평균 점수(`Stars`) |

※ `channel` 변수에 해상도(`'Pixel'` 문자 참조) 또는 운영체제 정보(`'Oper'` 문자 참조)가 있는 TV는 분석에서 제외하시오.  
※ Netflix와 Prime Video 제공여부 변수는 `channel` 변수를 참고하시오.  
※ 독립변수를 생성하면서 결측치가 생성되는 경우 해당 행을 모두 제거하시오.  
※ 학습 대상이 되는 행 개수는 197이다.  
※ 고해상도 여부 변수 생성은 해상도(`Picture_quality`) 변수만 참고하여 생성하시오.  
※ `seed`는 123으로 지정하시오. `(정답 예시: 고해상도여부 - 띄어쓰기 않고 문제에서 제시된 독립변수명을 기입)`

In [ ]:
df_q3 = df.copy()
cond_q3 = df_q3['channel'].str.contains('Pixel|Oper')
df_q3_drop = df_q3.loc[~cond_q3, :].copy()
display(df_q3.shape, df_q3_drop.shape)

df_q3_drop['R/R'] = df_q3_drop['Reviews'] / df_q3_drop['Ratings']
df_q3_drop['c/M'] = df_q3_drop['current_price'] / df_q3_drop['MRP']
display(df_q3_drop['channel'].value_counts())
df_q3_drop['Netflix'] = df_q3_drop['channel'].str.contains('Netflix').astype(int) #bool 2 int -> 0,1
df_q3_drop['PrimeVideo'] = df_q3_drop['channel'].str.contains('Prime Video').astype(int) #bool 2 int -> 0,1
display(df_q3_drop['Netflix'].value_counts(),df_q3_drop['PrimeVideo'].value_counts() )
df_q3_drop['HighQuality'] = df_q3_drop['Picture_quality'].str.contains('4K|8K').astype(int)

df_q3_drop2 = df_q3_drop.dropna()
display(df_q3_drop2.shape)

(666, 11)

(548, 11)

Netflix|Prime Video|Disney+Hotstar|Youtube             294
Netflix|Disney+Hotstar|Youtube                         171
Netflix|Youtube                                         28
Netflix|Prime Video|Youtube                             21
Prime Video|Disney+Hotstar|Youtube                      18
Disney+Hotstar|Youtube                                   6
Youtube                                                  4
Netflix|Prime Video|Apple TV|Disney+Hotstar|Youtube      4
Prime Video|Youtube                                      1
Prime Video                                              1
Name: channel, dtype: int64

1    518
0     30
Name: Netflix, dtype: int64

1    339
0    209
Name: PrimeVideo, dtype: int64

(197, 16)

In [ ]:
#DNME

#D
cols_X = ['R/R', 'MRP', 'c/M', 'Netflix', 'PrimeVideo', 'HighQuality']
cols_y = 'Stars' #['Stars'] 로하고 df_q3_drop2[cols_y] 는  df_q3_drop2[['Stars']] 라 series가 아닌 dataframe이 된다.

df_X = df_q3_drop2[cols_X].copy()
df_y = df_q3_drop2[cols_y].copy()
display(df_X.shape, df_y.shape)
#N

#M
model = RandomForestRegressor(random_state = 123)
model.fit(df_X, df_y) #randomforest의 y는 series 이길 원한다.
ser = pd.Series(dict(zip(cols_X, model.feature_importances_)))
display(ser)
display(ser.sort_values(ascending = False))
display(ser.idxmax())
#E

(197, 6)

(197,)

R/R            0.312206
MRP            0.173840
c/M            0.444691
Netflix        0.027141
PrimeVideo     0.024004
HighQuality    0.018118
dtype: float64

c/M            0.444691
R/R            0.312206
MRP            0.173840
Netflix        0.027141
PrimeVideo     0.024004
HighQuality    0.018118
dtype: float64

'c/M'